## Apply the Combined Scoring Matrix (Separately for Each Asset Group)

We apply the matrix **separately** for Asset 1 and Asset 2, using each group's
own vulnerability and risk scores.

This produces: `combined_a1` and `combined_a2`.

## 1. Setup & Libraries

We'll use the same libraries as throughout the course. In Google Colab, we mount your Google Drive and install additional GIS packages.


In [ ]:
# === ENVIRONMENT SETUP ===
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/MSER_510_VULNERABILTY'
    print("Google Drive connected!")
except Exception:
    BASE_DIR = './'
    print("Running locally.")

DATA_DIR = os.path.join(BASE_DIR, 'data')
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')

os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(OUTPUT_DIR, exist_ok=True)

# GeoPackage file paths for chained loading
INPUT_GPKG = os.path.join(DATA_DIR, 'class_7_risk.gpkg')  # Load from Class 7
OUTPUT_GPKG = os.path.join(DATA_DIR, 'class_8_final.gpkg')  # Save to Class 8
CLASS0_GPKG = os.path.join(DATA_DIR, 'vulnerability_risk_data.gpkg')  # Fallback for base layers

print(f"  Base directory: {BASE_DIR}")
print(f"  Data directory: {DATA_DIR}")
print(f"  Output directory: {OUTPUT_DIR}")

In [ ]:
# Install required packages
!pip install geopandas fiona shapely pyproj requests folium seaborn contextily --quiet

# Import libraries
import geopandas as gpd
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import folium
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap, Normalize
import os
import warnings
warnings.filterwarnings('ignore')

print("All libraries loaded successfully!")

## 2. Load Data with Vulnerability & Risk Scores

We load the parcels layer from the GeoPackage. This layer should contain:
- **vulnerability**: Score from Class 4 (1=Low, 2=Medium, 3=High)
- **risk**: Score from Class 7 (1=Low, 2=Medium, 3=High)
- All other fields from previous classes (exposure, potential_impact, adaptive_capacity, probability, consequence)

If you haven't completed Classes 4 and 7 yet, those fields will be populated by running those notebooks first.


In [ ]:
# Define path to GeoPackage (from Class 7)
gpkg_path = INPUT_GPKG

# Check if file exists
if os.path.exists(gpkg_path):
    print(f"✓ Found INPUT GeoPackage: {gpkg_path}")
else:
    print(f"✗ NOT FOUND: {gpkg_path}")
    print(f"   Make sure you've completed Classes 1-7 first!")

In [ ]:
# Load parcels layer from INPUT GeoPackage (from Class 7)
parcels = gpd.read_file(gpkg_path, layer='parcels')

print(f"Loaded {len(parcels)} parcels from Class 7")
print(f"\nColumns available:")
print(parcels.columns.tolist())

# Show first few rows
print(f"\nFirst 5 parcels:")
parcels.head()

In [ ]:
# Check for required vulnerability and risk fields (per asset group)
required_fields = ['vulnerability_a1', 'vulnerability_a2', 'risk_a1', 'risk_a2']
missing = [f for f in required_fields if f not in parcels.columns]

if missing:
    print(f"\u26a0 WARNING: Missing fields: {missing}")
    print("These will be created with default values for demonstration.")
    for field in missing:
        parcels[field] = 1  # Default to Low (1)
else:
    print(f"\u2713 All required fields present!")

# Display summary statistics for Asset 1
print(f"\n--- Asset 1 (is_asset_1 == 1) ---")
a1 = parcels[parcels['is_asset_1'] == 1] if 'is_asset_1' in parcels.columns else parcels
print(f"Vulnerability (A1) Distribution:")
print(a1['vulnerability_a1'].value_counts().sort_index())
print(f"\nRisk (A1) Distribution:")
print(a1['risk_a1'].value_counts().sort_index())

# Display summary statistics for Asset 2
print(f"\n--- Asset 2 (is_asset_2 == 1) ---")
a2 = parcels[parcels['is_asset_2'] == 1] if 'is_asset_2' in parcels.columns else parcels
print(f"Vulnerability (A2) Distribution:")
print(a2['vulnerability_a2'].value_counts().sort_index())
print(f"\nRisk (A2) Distribution:")
print(a2['risk_a2'].value_counts().sort_index())

## 3. Understanding the Combined Scoring Matrix

### How Vulnerability & Risk Come Together

In the previous classes, you learned:

- **Vulnerability (Class 4)**: Combines Potential Impact × Adaptive Capacity
  - High vulnerability = high impact + low adaptive capacity = more exposed to harm

- **Risk (Class 7)**: Combines Probability × Consequence
  - High risk = likely hazard + serious consequences = more likely to cause damage

The **Combined Score** brings these two perspectives together. It answers:

> *"How vulnerable is this location, AND how likely is it to experience damage?"*

### The Decision Matrix

We use a **2×3 matrix** to combine these scores:

| **Vulnerability** | **Risk: Low (1)** | **Risk: Medium (2)** | **Risk: High (3)** |
|---|---|---|---|
| **High (3)** | Medium (2) | High (3) | High (3) |
| **Medium (2)** | Low (1) | Medium (2) | High (3) |
| **Low (1)** | Low (1) | Low (1) | Medium (2) |

### Reading the Matrix

- **Low vulnerability + Low risk** = Safe area → Low Combined Score
- **High vulnerability + Low risk** = Vulnerable but protected → Medium Combined Score
- **High vulnerability + High risk** = Critical area → High Combined Score

The matrix emphasizes that **high risk + high vulnerability is most concerning**, but **even medium vulnerability can become high priority if risk is high**.


## 4. Visualize the Scoring Matrix

This heatmap shows **how vulnerability and risk combine**. This is the key visual for understanding the combined assessment.

**How to read it:**
- Find your parcel's Vulnerability score (rows) and Risk score (column)
- The cell shows the Combined Score
- Colors show the severity: lighter = lower score, darker = higher score


In [ ]:
# Define the combined scoring matrix
combined_matrix_dict = {
    (3, 1): 2,  # High vuln, Low risk → Medium
    (3, 2): 3,  # High vuln, Medium risk → High
    (3, 3): 3,  # High vuln, High risk → High
    (2, 1): 1,  # Medium vuln, Low risk → Low
    (2, 2): 2,  # Medium vuln, Medium risk → Medium
    (2, 3): 3,  # Medium vuln, High risk → High
    (1, 1): 1,  # Low vuln, Low risk → Low
    (1, 2): 1,  # Low vuln, Medium risk → Low
    (1, 3): 2,  # Low vuln, High risk → Medium
}

# Create the matrix as a 2D array for visualization
# Rows: Vulnerability (3=High, 2=Medium, 1=Low) - high at top
# Cols: Risk (1=Low, 2=Medium, 3=High)
matrix_data = np.array([
    [combined_matrix_dict[(3, 1)], combined_matrix_dict[(3, 2)], combined_matrix_dict[(3, 3)]],
    [combined_matrix_dict[(2, 1)], combined_matrix_dict[(2, 2)], combined_matrix_dict[(2, 3)]],
    [combined_matrix_dict[(1, 1)], combined_matrix_dict[(1, 2)], combined_matrix_dict[(1, 3)]],
])

print("Combined Scoring Matrix:")
print(matrix_data)


In [ ]:
# Create the heatmap visualization
fig, ax = plt.subplots(figsize=(10, 7))

# Define colors: 1=light peach, 2=salmon, 3=strong red (official combined colors)
colors = ['#F0D5C4', '#D07872', '#C23B33']
cmap = ListedColormap(colors)

# Create the heatmap
im = ax.imshow(matrix_data, cmap=cmap, aspect='auto', vmin=1, vmax=3)

# Set row and column labels
vuln_labels = ['High (3)', 'Medium (2)', 'Low (1)']
risk_labels = ['Low (1)', 'Medium (2)', 'High (3)']

ax.set_xticks(np.arange(3))
ax.set_yticks(np.arange(3))
ax.set_xticklabels(risk_labels, fontsize=12, fontweight='bold')
ax.set_yticklabels(vuln_labels, fontsize=12, fontweight='bold')

# Add labels
ax.set_xlabel('RISK SCORE', fontsize=13, fontweight='bold', labelpad=15)
ax.set_ylabel('VULNERABILITY SCORE', fontsize=13, fontweight='bold', labelpad=15)

# Add text annotations in each cell
for i in range(3):
    for j in range(3):
        value = matrix_data[i, j]
        # Choose text color based on background
        text_color = 'white' if value >= 2 else 'black'
        text = ax.text(j, i, f'{int(value)}',
                      ha="center", va="center", color=text_color,
                      fontsize=20, fontweight='bold')

# Add title
plt.title('Combined Vulnerability & Risk Scoring Matrix',
          fontsize=16, fontweight='bold', pad=20)

# Add grid for clarity
ax.set_xticks(np.arange(3) - 0.5, minor=True)
ax.set_yticks(np.arange(3) - 0.5, minor=True)
ax.grid(which='minor', color='black', linestyle='-', linewidth=2)

plt.tight_layout()
plt.show()

print("✓ Matrix visualization complete!")
print("\nHow to read this matrix:")
print("- Row: Find your parcel's Vulnerability score (High/Medium/Low)")
print("- Column: Find your parcel's Risk score (Low/Medium/High)")
print("- Cell Value: The Combined Score (1=Low, 2=Medium, 3=High)")

## 5. Calculate Combined Scores

Now we apply the matrix to every parcel in our dataset. For each parcel, we:

1. Look up its Vulnerability score (from Class 4)
2. Look up its Risk score (from Class 7)
3. Use the matrix to find the Combined Score

This is a simple lookup operation that brings together all the analysis we've done.


In [ ]:
# Apply the combined scoring matrix separately for each asset group
parcels['combined_a1'] = parcels.apply(
    lambda row: combined_matrix_dict.get((int(row['vulnerability_a1']), int(row['risk_a1'])), 0),
    axis=1
)
parcels['combined_a2'] = parcels.apply(
    lambda row: combined_matrix_dict.get((int(row['vulnerability_a2']), int(row['risk_a2'])), 0),
    axis=1
)

print("Combined scores calculated for both asset groups!")
score_labels = {3: 'High', 2: 'Medium', 1: 'Low', 0: 'No Data'}
for col, label in [('combined_a1', 'Asset 1'), ('combined_a2', 'Asset 2')]:
    asset_flag = 'is_asset_1' if 'a1' in col else 'is_asset_2'
    in_group = parcels[parcels[asset_flag] == 1]
    print(f"\n{label} ({len(in_group):,} parcels):")
    for score in sorted(in_group[col].unique(), reverse=True):
        if score == 0:
            continue
        count = (in_group[col] == score).sum()
        pct = 100 * count / len(in_group)
        print(f"  {score_labels.get(score, '?')} ({score}): {count:,} ({pct:.1f}%)")

## 5.5. Detailed Cross-Tabulation Results Matrix

### Combined Vulnerability & Risk Results Analysis

This section presents a **detailed cross-tabulation matrix** showing the results of applying the combined scoring matrix to actual parcel data.

For each combination of Vulnerability and Risk scores, we analyze:
- **Number of Parcels**: How many properties fall into each vulnerability-risk combination
- **Parcel Value**: Total assessed value of land parcels
- **Building Value**: Total assessed value of structures
- **Combined Score**: The resulting priority score from our decision matrix

This reveals which vulnerability-risk combinations have the most properties and assets at stake, helping decision-makers prioritize investment and intervention.

**Color Coding:**
- Light peach background: Low combined score (1) = Lower priority
- Salmon background: Medium combined score (2) = Moderate priority  
- Red background: High combined score (3) = High priority


In [ ]:
# Helper function to format currency
def format_currency(value):
    """Format numeric value as currency string."""
    if pd.isna(value) or value == 0:
        return "$0"
    abs_val = abs(value)
    if abs_val >= 1e6:
        return f"${value/1e6:.1f}M"
    elif abs_val >= 1e3:
        return f"${value/1e3:.1f}K"
    else:
        return f"${value:.0f}"

# Build detailed results matrices for BOTH asset groups
asset_configs = [
    {'label': 'Asset 1', 'vuln': 'vulnerability_a1', 'risk': 'risk_a1',
     'flag': 'is_asset_1', 'combined': 'combined_a1'},
    {'label': 'Asset 2', 'vuln': 'vulnerability_a2', 'risk': 'risk_a2',
     'flag': 'is_asset_2', 'combined': 'combined_a2'},
]

for asset_cfg in asset_configs:
    # Filter to relevant parcels
    if asset_cfg['flag'] in parcels.columns:
        asset_parcels = parcels[parcels[asset_cfg['flag']] == 1].copy()
    else:
        asset_parcels = parcels.copy()

    results_matrix_data = []
    for vuln in [3, 2, 1]:
        row_data = []
        for risk in [1, 2, 3]:
            combo_parcels = asset_parcels[
                (asset_parcels[asset_cfg['vuln']] == vuln) &
                (asset_parcels[asset_cfg['risk']] == risk)
            ]
            count = len(combo_parcels)

            parval_sum = 0
            if 'parcel_value' in combo_parcels.columns:
                parval_sum = combo_parcels['parcel_value'].fillna(0).sum()
            elif 'parval' in combo_parcels.columns:
                parval_sum = combo_parcels['parval'].fillna(0).sum()

            improvval_sum = combo_parcels['structure_value'].sum() if 'structure_value' in combo_parcels.columns else 0

            combined = combined_matrix_dict.get((vuln, risk), 0)

            row_data.append({
                'vuln': vuln, 'risk': risk, 'combined': combined,
                'count': count, 'parval': parval_sum, 'improvval': improvval_sum
            })
        results_matrix_data.append(row_data)

    # --- Visualization ---
    fig, ax = plt.subplots(figsize=(14, 11))

    color_map_results = {1: '#F0D5C4', 2: '#D07872', 3: '#C23B33'}

    cell_width = 1.0
    cell_height = 1.0
    cell_spacing = 0.1
    y_base = 2.5

    risk_labels = ['Risk: Low (1)', 'Risk: Medium (2)', 'Risk: High (3)']
    for col_idx, risk_label in enumerate(risk_labels):
        x = col_idx * (cell_width + cell_spacing) + 1.5
        ax.text(x + cell_width/2, y_base + 0.3, risk_label,
               ha='center', va='center', fontsize=11, fontweight='bold')

    vuln_labels = ['Vuln: High (3)', 'Vuln: Medium (2)', 'Vuln: Low (1)']
    for row_idx, (vuln_label, row_cells) in enumerate(zip(vuln_labels, results_matrix_data)):
        y = y_base - (row_idx + 1) * (cell_height + cell_spacing)
        ax.text(0.5, y + cell_height/2, vuln_label,
               ha='center', va='center', fontsize=11, fontweight='bold')

        for col_idx, cell_data in enumerate(row_cells):
            x = col_idx * (cell_width + cell_spacing) + 1.5
            bg_color = color_map_results.get(cell_data['combined'], '#FFFFFF')
            text_color = 'white' if cell_data['combined'] >= 2 else 'black'

            from matplotlib.patches import Rectangle
            rect = Rectangle((x, y), cell_width, cell_height,
                             linewidth=2, edgecolor='black', facecolor=bg_color)
            ax.add_patch(rect)

            combined_name = {1: 'Low', 2: 'Medium', 3: 'High'}.get(cell_data['combined'], '?')
            cell_text = f"{combined_name} ({cell_data['combined']})\n"
            cell_text += f"{cell_data['count']} parcels\n"
            cell_text += f"{format_currency(cell_data['parval'])} parcel val\n"
            cell_text += f"{format_currency(cell_data['improvval'])} bldg val"

            ax.text(x + cell_width/2, y + cell_height/2, cell_text,
                   ha='center', va='center', fontsize=9, fontweight='bold',
                   color=text_color, linespacing=1.4)

    ax.set_xlim(0, 5)
    ax.set_ylim(-0.5, 3.5)
    ax.axis('off')

    plt.suptitle(f'Combined Vulnerability & Risk Results Matrix \u2014 {asset_cfg["label"]}',
                fontsize=16, fontweight='bold', y=0.98)
    plt.tight_layout()
    plt.show()

    print(f"\u2713 Results matrix for {asset_cfg['label']} complete!")
    print("\nMatrix Summary:")
    print("=" * 100)
    for row in results_matrix_data:
        for cell in row:
            if cell['count'] > 0:
                v_name = {1: 'Low', 2: 'Med', 3: 'High'}[cell['vuln']]
                r_name = {1: 'Low', 2: 'Med', 3: 'High'}[cell['risk']]
                c_name = {1: 'Low', 2: 'Med', 3: 'High'}[cell['combined']]
                print(f"Vuln:{v_name} + Risk:{r_name} \u2192 Combined:{c_name} | "
                      f"{cell['count']:4d} parcels | Parcel: {format_currency(cell['parval']):>10s} | "
                      f"Bldg: {format_currency(cell['improvval']):>10s}")
    print("=" * 100)
    print()

## 5.6. Grand Summary: All 8 Scores Side-by-Side

### The Complete Analysis Journey

This culminating summary table brings together all 8 scores from the entire course, showing how each metric is distributed across your study area.

For each score (exposure, potential_impact, adaptive_capacity, vulnerability, probability, consequence, risk, and vuln_risk_combined), we show:
- **High Count**: Number of parcels with high score (3)
- **High %**: Percentage of parcels with high score
- **Medium Count**: Number of parcels with medium score (2)
- **Medium %**: Percentage of parcels with medium score
- **Low Count**: Number of parcels with low score (1)
- **Low %**: Percentage of parcels with low score
- **High Value**: Total monetary value of high-scoring parcels (to show asset concentration)

This table makes it easy to compare how different aspects of vulnerability and risk are distributed, and reveals which metrics are most concerning for your study area.


In [ ]:
# Build comprehensive summary table of all 8 scores (per asset group)
asset_groups = [
    {'label': 'Asset 1', 'flag': 'is_asset_1',
     'fields': ['exposure_a1', 'potential_impact_a1', 'adaptive_capacity_a1', 'vulnerability_a1',
                'probability_a1', 'consequence_a1', 'risk_a1', 'combined_a1']},
    {'label': 'Asset 2', 'flag': 'is_asset_2',
     'fields': ['exposure_a2', 'potential_impact_a2', 'adaptive_capacity_a2', 'vulnerability_a2',
                'probability_a2', 'consequence_a2', 'risk_a2', 'combined_a2']},
]

# Determine which value field to use
value_field = None
for candidate in ['parcel_value', 'parval', 'improvement_value', 'improvval', 'structure_value', 'total_value']:
    if candidate in parcels.columns:
        value_field = candidate
        break

for ag in asset_groups:
    if ag['flag'] in parcels.columns:
        asset_parcels = parcels[parcels[ag['flag']] == 1]
    else:
        asset_parcels = parcels

    grand_summary = []
    for field in ag['fields']:
        if field not in asset_parcels.columns:
            continue

        n = len(asset_parcels)
        high_count = (asset_parcels[field] == 3).sum()
        high_pct = (high_count / n) * 100 if n > 0 else 0
        med_count = (asset_parcels[field] == 2).sum()
        med_pct = (med_count / n) * 100 if n > 0 else 0
        low_count = (asset_parcels[field] == 1).sum()
        low_pct = (low_count / n) * 100 if n > 0 else 0

        high_value = 0
        if value_field and high_count > 0:
            high_value = asset_parcels[asset_parcels[field] == 3][value_field].fillna(0).sum()

        grand_summary.append({
            'Score': field.replace('_', ' ').title(),
            'High Count': high_count,
            'High %': f"{high_pct:.1f}%",
            'Med Count': med_count,
            'Med %': f"{med_pct:.1f}%",
            'Low Count': low_count,
            'Low %': f"{low_pct:.1f}%",
            'High Value': format_currency(high_value) if value_field else 'N/A'
        })

    grand_summary_df = pd.DataFrame(grand_summary)

    print("\n" + "=" * 160)
    print(f"GRAND SUMMARY: ALL 8 ASSESSMENT SCORES — {ag['label']}")
    print("=" * 160)
    print(grand_summary_df.to_string(index=False))
    print("=" * 160)

    # Export to CSV
    OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    csv_path = os.path.join(OUTPUT_DIR, f'grand_summary_all_scores_{ag["label"].lower().replace(" ", "_")}.csv')
    grand_summary_df.to_csv(csv_path, index=False)
    print(f"\n✓ Grand summary exported to: {csv_path}")
    display(grand_summary_df)

## 6. Summary Statistics by Combined Score

Let's see how our results are distributed and what values are at risk.


In [ ]:
# Create summary table of all 8 scores (per asset group)
asset_groups = [
    {'label': 'Asset 1', 'flag': 'is_asset_1',
     'fields': ['exposure_a1', 'potential_impact_a1', 'adaptive_capacity_a1', 'vulnerability_a1',
                'probability_a1', 'consequence_a1', 'risk_a1', 'combined_a1']},
    {'label': 'Asset 2', 'flag': 'is_asset_2',
     'fields': ['exposure_a2', 'potential_impact_a2', 'adaptive_capacity_a2', 'vulnerability_a2',
                'probability_a2', 'consequence_a2', 'risk_a2', 'combined_a2']},
]

for ag in asset_groups:
    if ag['flag'] in parcels.columns:
        asset_parcels = parcels[parcels[ag['flag']] == 1]
    else:
        asset_parcels = parcels

    summary_data = []
    for field in ag['fields']:
        if field in asset_parcels.columns:
            n = len(asset_parcels)
            high = (asset_parcels[field] == 3).sum()
            high_pct = (high / n) * 100 if n > 0 else 0
            med = (asset_parcels[field] == 2).sum()
            med_pct = (med / n) * 100 if n > 0 else 0
            low = (asset_parcels[field] == 1).sum()
            low_pct = (low / n) * 100 if n > 0 else 0

            summary_data.append({
                'Score': field.replace('_', ' ').title(),
                'High (3)': f"{high} ({high_pct:.1f}%)",
                'Medium (2)': f"{med} ({med_pct:.1f}%)",
                'Low (1)': f"{low} ({low_pct:.1f}%)",
            })

    summary_df = pd.DataFrame(summary_data)
    print(f"\nASSESSMENT RESULTS ACROSS ALL CLASSES — {ag['label']}:")
    print("=" * 120)
    print(summary_df.to_string(index=False))
    print("=" * 120)

## 7. Comprehensive Multi-Panel Visualization

This figure shows all 8 scores in one place, so you can see how the assessment evolved from Exposure → Combined Score.


In [ ]:
# Create multi-panel figures showing all 8 scores per asset group
asset_groups = [
    {'label': 'Asset 1', 'flag': 'is_asset_1',
     'fields': ['exposure_a1', 'potential_impact_a1', 'adaptive_capacity_a1', 'vulnerability_a1',
                'probability_a1', 'consequence_a1', 'risk_a1', 'combined_a1']},
    {'label': 'Asset 2', 'flag': 'is_asset_2',
     'fields': ['exposure_a2', 'potential_impact_a2', 'adaptive_capacity_a2', 'vulnerability_a2',
                'probability_a2', 'consequence_a2', 'risk_a2', 'combined_a2']},
]

colors_map = {1: '#F0D5C4', 2: '#D07872', 3: '#C23B33'}

for ag in asset_groups:
    if ag['flag'] in parcels.columns:
        asset_parcels = parcels[parcels[ag['flag']] == 1]
    else:
        asset_parcels = parcels

    fig, axes = plt.subplots(2, 4, figsize=(18, 8))
    axes = axes.flatten()

    for idx, (ax, field) in enumerate(zip(axes, ag['fields'])):
        if field in asset_parcels.columns:
            counts = asset_parcels[field].value_counts().sort_index()
            score_names = ['Low (1)', 'Medium (2)', 'High (3)']
            values = [counts.get(i, 0) for i in [1, 2, 3]]
            bar_colors = [colors_map.get(i, 'gray') for i in [1, 2, 3]]

            bars = ax.bar(score_names, values, color=bar_colors, edgecolor='black', linewidth=1.5)

            for bar in bars:
                height = bar.get_height()
                if height > 0:
                    ax.text(bar.get_x() + bar.get_width()/2., height,
                           f'{int(height)}',
                           ha='center', va='bottom', fontweight='bold')

            ax.set_ylabel('Number of Parcels', fontweight='bold')
            title = field.replace('_', ' ').title()
            ax.set_title(title, fontweight='bold', fontsize=11)
            ax.set_ylim(0, max(values) * 1.15 if max(values) > 0 else 1)
            ax.grid(axis='y', alpha=0.3)

    plt.suptitle(f'Complete Assessment: All 8 Scores — {ag["label"]}',
                 fontsize=16, fontweight='bold', y=1.00)
    plt.tight_layout()
    plt.show()

print("✓ Multi-panel visualization complete!")

In [ ]:
# Create cross-tabulation: Vulnerability vs Risk vs Combined (per asset group)
asset_configs = [
    {'label': 'Asset 1', 'flag': 'is_asset_1',
     'vuln': 'vulnerability_a1', 'risk': 'risk_a1', 'combined': 'combined_a1'},
    {'label': 'Asset 2', 'flag': 'is_asset_2',
     'vuln': 'vulnerability_a2', 'risk': 'risk_a2', 'combined': 'combined_a2'},
]

for ac in asset_configs:
    if ac['flag'] in parcels.columns:
        asset_parcels = parcels[parcels[ac['flag']] == 1]
    else:
        asset_parcels = parcels

    crosstab_data = []
    for vuln in sorted(asset_parcels[ac['vuln']].dropna().unique()):
        for risk in sorted(asset_parcels[ac['risk']].dropna().unique()):
            subset = asset_parcels[
                (asset_parcels[ac['vuln']] == vuln) & (asset_parcels[ac['risk']] == risk)
            ]
            if len(subset) > 0:
                combined = subset[ac['combined']].iloc[0]
                crosstab_data.append({
                    'Vulnerability': int(vuln),
                    'Risk': int(risk),
                    'Combined Score': int(combined),
                    'Count': len(subset),
                    'Percentage': f"{(len(subset)/len(asset_parcels)*100):.1f}%"
                })

    crosstab_df = pd.DataFrame(crosstab_data)
    print(f"\nCross-Tabulation: Vulnerability \u00d7 Risk \u2192 Combined Score \u2014 {ac['label']}")
    print("=" * 90)
    print(crosstab_df.to_string(index=False))
    print("=" * 90)

## 8. Map the Final Combined Score

We'll create a beautiful map showing the Combined Score across the study area. This is the final, most important map of the course.


In [ ]:
# Load flood zones for reference (if available)
try:
    flood_zones = gpd.read_file(gpkg_path, layer='flood_zones')
    print(f"✓ Loaded {len(flood_zones)} flood zone features")
except:
    print("⚠ Flood zones layer not found")
    flood_zones = None


In [ ]:
import contextily as ctx
from matplotlib.patches import Patch
import matplotlib.gridspec as gridspec

# Reproject to Web Mercator for basemap tiles
parcels_wm = parcels.to_crs(epsg=3857)

combined_colors = {1: '#F0D5C4', 2: '#D07872', 3: '#C23B33'}

# Create side-by-side figure: Asset 1 (left) and Asset 2 (right)
fig = plt.figure(figsize=(20, 12))
gs = gridspec.GridSpec(2, 2, height_ratios=[10, 1.2], hspace=0.05, wspace=0.05)
ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])
ax_legend = fig.add_subplot(gs[1, :])

for ax, asset_label, flag_col, combined_col in [
    (ax1, 'Asset 1', 'is_asset_1', 'combined_a1'),
    (ax2, 'Asset 2', 'is_asset_2', 'combined_a2'),
]:
    # All parcels as background
    parcels_wm.plot(ax=ax, facecolor='none', edgecolor='#888888', linewidth=0.3)

    # Filter to asset group
    if flag_col in parcels_wm.columns:
        asset_wm = parcels_wm[parcels_wm[flag_col] == 1]
    else:
        asset_wm = parcels_wm

    # Combined score layer
    for score in [3, 2, 1]:
        subset = asset_wm[asset_wm[combined_col] == score]
        if len(subset) > 0:
            subset.plot(ax=ax, facecolor=combined_colors[score], edgecolor='none', alpha=0.85)

    # Buildings
    try:
        buildings_layer = gpd.read_file(CLASS0_GPKG, layer='buildings')
        buildings_wm = buildings_layer.to_crs(epsg=3857)
        buildings_wm.plot(ax=ax, facecolor='#3D3D3D', edgecolor='#2a2a2a', linewidth=0.1, alpha=0.7)
    except Exception:
        pass

    # Flood zones
    try:
        flood_zones_full = gpd.read_file(CLASS0_GPKG, layer='flood_zones')
        flood_wm = flood_zones_full.to_crs(epsg=3857)
        flood_colors = {'Floodway': '#2B5797', '100-year': '#8FABBE', '500-year': '#B4D4E7'}
        for flood_type in ['500-year', '100-year', 'Floodway']:
            flood_subset = flood_wm[flood_wm['flood_category'] == flood_type]
            if len(flood_subset) > 0:
                flood_subset.plot(ax=ax, facecolor=flood_colors.get(flood_type, '#B4D4E7'),
                                edgecolor='none', alpha=0.5)
    except Exception:
        pass

    # Basemap
    ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron, zoom='auto')
    ax.set_axis_off()
    ax.set_title(f'Combined Vulnerability & Risk \u2014 {asset_label}',
                fontsize=14, fontweight='bold', pad=10)

# Legend panel
ax_legend.set_axis_off()
legend_elements = [
    Patch(facecolor='#C23B33', edgecolor='none', label='High (3)'),
    Patch(facecolor='#D07872', edgecolor='none', label='Medium (2)'),
    Patch(facecolor='#F0D5C4', edgecolor='none', label='Low (1)'),
    Patch(facecolor='none', edgecolor='none', label=''),
    Patch(facecolor='#3D3D3D', edgecolor='#2a2a2a', label='Buildings'),
    Patch(facecolor='#2B5797', edgecolor='none', alpha=0.5, label='Floodway'),
    Patch(facecolor='#8FABBE', edgecolor='none', alpha=0.5, label='100-year Floodplain'),
    Patch(facecolor='#B4D4E7', edgecolor='none', alpha=0.5, label='500-year Floodplain'),
    Patch(facecolor='none', edgecolor='#888888', linewidth=0.5, label='Parcels'),
]
ax_legend.legend(handles=legend_elements, loc='center', ncol=4, fontsize=9,
                frameon=True, facecolor='white', edgecolor='#cccccc',
                handlelength=1.5, handletextpad=0.5, columnspacing=1.5)

# Export
OUTPUT_DIR = os.path.join(BASE_DIR, 'outputs')
os.makedirs(OUTPUT_DIR, exist_ok=True)
png_path = os.path.join(OUTPUT_DIR, 'combined_score_map.png')
plt.savefig(png_path, dpi=150, bbox_inches='tight', facecolor='white', edgecolor='none')
plt.show()
print(f"\u2713 Side-by-side maps exported to: {png_path}")

## Step 8.5: Export Combined Score Map as PNG

Now we'll create a publication-quality PNG map that combines all layers in the proper order with dark styling. This map shows the final combined vulnerability and risk assessment with all supporting layers (buildings, flood zones, and parcels) in a single exportable image.

## 9. Asset Values at Risk

Now let's calculate which areas have the most value at risk based on the combined score.


In [ ]:
# Check for value fields
value_fields = [col for col in parcels.columns if 'value' in col.lower()]
print(f"Available value fields: {value_fields}")

asset_configs = [
    {'label': 'Asset 1', 'flag': 'is_asset_1', 'combined': 'combined_a1'},
    {'label': 'Asset 2', 'flag': 'is_asset_2', 'combined': 'combined_a2'},
]

if len(value_fields) > 0:
    value_col = value_fields[0]
    print(f"\nUsing field: {value_col}")

    for ac in asset_configs:
        if ac['flag'] in parcels.columns:
            asset_parcels = parcels[parcels[ac['flag']] == 1]
        else:
            asset_parcels = parcels

        print(f"\nTotal Assets at Risk by Combined Score \u2014 {ac['label']}:")
        print("=" * 70)

        for score in sorted(asset_parcels[ac['combined']].dropna().unique()):
            if score == 0:
                continue
            subset = asset_parcels[asset_parcels[ac['combined']] == score]
            count = len(subset)
            total_value = subset[value_col].sum()
            avg_value = subset[value_col].mean()
            score_name = {1: 'Low', 2: 'Medium', 3: 'High'}.get(int(score), 'Unknown')

            print(f"{score_name} ({int(score)}): {count:5d} parcels | "
                  f"Total Value: ${total_value:>15,.0f} | Avg: ${avg_value:>12,.0f}")

        print("=" * 70)
else:
    print("\u26a0 No value field found. Skipping asset analysis.")

## 10. Complete Field Reference Guide

Here's what every field means and which class created it:


In [ ]:
# Create field reference table
field_ref = pd.DataFrame([
    {
        'Field': 'exposure_a1 / _a2',
        'Class': 'Class 1',
        'Score Range': '0-1',
        'Description': 'Is the parcel in the 100-year flood zone? Per asset group (0=Not exposed/not in group, 1=Exposed)'
    },
    {
        'Field': 'potential_impact_a1 / _a2',
        'Class': 'Class 2',
        'Score Range': '0-3',
        'Description': 'What is exposed? Scored per asset group (0=Not in group, 1=Low, 3=High impact)'
    },
    {
        'Field': 'adaptive_capacity_a1 / _a2',
        'Class': 'Class 3',
        'Score Range': '0-3',
        'Description': 'How prepared are we? Year built vs building codes per asset group (0=Not in group, 1=Low, 3=High)'
    },
    {
        'Field': 'vulnerability_a1 / _a2',
        'Class': 'Class 4',
        'Score Range': '0-3',
        'Description': 'Combined Potential Impact x Adaptive Capacity per asset group (0=Not in group, 1=Low, 3=High)'
    },
    {
        'Field': 'probability_a1 / _a2',
        'Class': 'Class 5',
        'Score Range': '0-3',
        'Description': 'Flood probability by zone type per asset group (0=Not in group, 1=Low, 3=High)'
    },
    {
        'Field': 'consequence_a1 / _a2',
        'Class': 'Class 6',
        'Score Range': '0-3',
        'Description': 'Potential consequence by structure value per asset group (0=Not in group, 1=Low, 3=High)'
    },
    {
        'Field': 'risk_a1 / _a2',
        'Class': 'Class 7',
        'Score Range': '0-3',
        'Description': 'Combined Probability x Consequence per asset group (0=Not in group, 1=Low, 3=High)'
    },
    {
        'Field': 'combined_a1 / _a2',
        'Class': 'Class 8',
        'Score Range': '0-3',
        'Description': 'Combined Vulnerability x Risk using matrix per asset group (0=Not in group, 1=Low, 3=High)'
    },
])

print("COMPLETE FIELD REFERENCE GUIDE")
print("=" * 130)
for idx, row in field_ref.iterrows():
    print(f"\n{row['Field'].upper()}")
    print(f"  Created in: {row['Class']}")
    print(f"  Score Range: {row['Score Range']}")
    print(f"  Description: {row['Description']}")
print("\n" + "=" * 130)

## 11. Save Updated Data

We save the complete results back to the GeoPackage, including all 8 scores for future reference.


In [ ]:
import fiona
import sqlite3
import os

# Delete old output to prevent append-duplicates
if os.path.exists(OUTPUT_GPKG):
    os.remove(OUTPUT_GPKG)

try:
    # Save parcels layer (creates new GeoPackage)
    parcels.to_file(OUTPUT_GPKG, layer='parcels', driver='GPKG')
    print(f"✓ Saved parcels layer ({len(parcels)} features)")

    # Copy forward all other layers from the input GeoPackage
    if os.path.exists(INPUT_GPKG):
        input_layers = fiona.listlayers(INPUT_GPKG)
        output_layers = fiona.listlayers(OUTPUT_GPKG)
        for layer_name in input_layers:
            if layer_name == 'parcels' or layer_name in output_layers:
                continue
            try:
                layer_data = gpd.read_file(INPUT_GPKG, layer=layer_name)
                layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                output_layers.append(layer_name)
                print(f"✓ Copied layer: {layer_name} ({len(layer_data)} features)")
            except Exception as e:
                print(f"  Note: Could not copy layer '{layer_name}': {e}")

        # Copy non-spatial tables via sqlite3
        try:
            conn_in = sqlite3.connect(INPUT_GPKG)
            conn_out = sqlite3.connect(OUTPUT_GPKG)
            cursor = conn_in.cursor()
            cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
            all_tables = [row[0] for row in cursor.fetchall()]
            system_tables = ['gpkg_contents', 'gpkg_geometry_columns', 'gpkg_spatial_ref_sys',
                            'gpkg_ogr_contents', 'gpkg_tile_matrix', 'gpkg_tile_matrix_set',
                            'sqlite_sequence', 'gpkg_extensions', 'gpkg_metadata',
                            'gpkg_metadata_reference']
            for table in all_tables:
                if table in system_tables or table in input_layers or table.startswith('rtree_') or table.startswith('trigger_'):
                    continue
                try:
                    df = pd.read_sql(f'SELECT * FROM "{table}"', conn_in)
                    if len(df) > 0:
                        df.to_sql(table, conn_out, if_exists='replace', index=False)
                        print(f"✓ Copied non-spatial table: {table} ({len(df)} rows)")
                except Exception:
                    pass
            conn_in.close()
            conn_out.close()
        except Exception:
            pass

    # Ensure base layers from Class 0 are included
    if os.path.exists(CLASS0_GPKG):
        try:
            output_layers = fiona.listlayers(OUTPUT_GPKG)
            class0_layers = fiona.listlayers(CLASS0_GPKG)
            for layer_name in class0_layers:
                if layer_name not in output_layers:
                    try:
                        layer_data = gpd.read_file(CLASS0_GPKG, layer=layer_name)
                        layer_data.to_file(OUTPUT_GPKG, layer=layer_name, driver='GPKG', mode='a')
                        print(f"✓ Added base layer from Class 0: {layer_name} ({len(layer_data)} features)")
                    except Exception as e:
                        print(f"  Note: Could not copy base layer '{layer_name}': {e}")
        except Exception:
            pass

    print(f"\n✓ All data saved to: {OUTPUT_GPKG}")
except Exception as e:
    print(f"✗ Error saving: {e}")

In [ ]:
# Export summary statistics to CSV for easy reference
export_cols = ['objectid']

# Add asset-group-specific score columns
for col in ['vulnerability_a1', 'vulnerability_a2', 'risk_a1', 'risk_a2',
            'combined_a1', 'combined_a2', 'is_asset_1', 'is_asset_2']:
    if col in parcels.columns:
        export_cols.append(col)

# Add any other useful columns
for col in ['parcel_id', 'address', 'structure_value']:
    if col in parcels.columns:
        export_cols.append(col)

summary_export = parcels[[c for c in export_cols if c in parcels.columns]].copy()

# Sort by combined_a1 score (high to low), then combined_a2
sort_cols = [c for c in ['combined_a1', 'combined_a2'] if c in summary_export.columns]
if sort_cols:
    summary_export = summary_export.sort_values(sort_cols, ascending=False)

csv_path = os.path.join(OUTPUT_DIR, 'combined_scores_summary.csv')
summary_export.to_csv(csv_path, index=False)
print(f"\u2713 Exported summary to: {csv_path}")
print(f"\nFirst 20 parcels (sorted by combined score, highest first):")
print(summary_export.head(20).to_string(index=False))

## Summary

You've completed **Class 8: Combined Vulnerability & Risk Score**.

**What You Accomplished:**
- Applied the combined scoring matrix (Vulnerability x Risk) **separately** for Asset 1 and Asset 2
- Each asset group has its own final score: `combined_a1`, `combined_a2`
- Visualized both asset groups on a single map

**All 8 scores per asset group:**
1. Exposure (Class 1)
2. Potential Impact (Class 2) — `potential_impact_a1`, `potential_impact_a2`
3. Adaptive Capacity (Class 3) — `adaptive_capacity_a1`, `adaptive_capacity_a2`
4. Vulnerability (Class 4) — `vulnerability_a1`, `vulnerability_a2`
5. Probability (Class 5) — `probability_a1`, `probability_a2`
6. Consequence (Class 6) — `consequence_a1`, `consequence_a2`
7. Risk (Class 7) — `risk_a1`, `risk_a2`
8. Combined Score (Class 8) — `combined_a1`, `combined_a2`

## Color Reference for GIS Symbology

Use these hex color values when styling your combined score layer in QGIS or ArcGIS Pro:

| Score | Label | Hex Code | RGB |
|-------|-------|----------|-----|
| 1 | Low | `#F0D5C4` | 240, 213, 196 |
| 2 | Medium | `#D07872` | 208, 120, 114 |
| 3 | High | `#C23B33` | 194, 59, 51 |

**How to apply in QGIS:**
1. Right-click your layer → Properties → Symbology
2. Choose "Categorized" from the dropdown
3. Set Column to `combined_a1` / `combined_a2`
4. Click "Classify"
5. Double-click each symbol to change its color using the hex values above

**How to apply in ArcGIS Pro:**
1. Right-click your layer → Symbology
2. Choose "Unique Values"
3. Set Field 1 to `combined_a1` / `combined_a2`
4. Click "Add all values"
5. Double-click each symbol to change its color using the hex values above

**Pre-made symbology files** are also available in the `symbology/` folder:
- `combined_vuln_risk_symbology.qml` — Load in QGIS via Style → Load Style
- `combined_vuln_risk_symbology.lyrx` — Import in ArcGIS Pro via Symbology → Import


> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## 13. Using Results in QGIS & ArcGIS Pro

The analysis you did in Python can be replicated in other GIS software. Here's how:

### QGIS: Field Calculator Expression

To create the combined score field in QGIS:

1. Open the parcels layer in QGIS
2. Enter Edit Mode (pencil icon)
3. Open the Attribute Table
4. Create a new field called `combined_a1` / `combined_a2` (type: Integer)
5. Use the Field Calculator (click the calculator icon in the header)
6. Enter this expression:

```qgis
CASE
  WHEN "vulnerability_a1" = 3 AND "risk_a1" = 1 THEN 2
  WHEN "vulnerability_a1" = 3 AND "risk_a1" = 2 THEN 3
  WHEN "vulnerability_a1" = 3 AND "risk_a1" = 3 THEN 3
  WHEN "vulnerability_a1" = 2 AND "risk_a1" = 1 THEN 1
  WHEN "vulnerability_a1" = 2 AND "risk_a1" = 2 THEN 2
  WHEN "vulnerability_a1" = 2 AND "risk_a1" = 3 THEN 3
  WHEN "vulnerability_a1" = 1 AND "risk_a1" = 1 THEN 1
  WHEN "vulnerability_a1" = 1 AND "risk_a1" = 2 THEN 1
  WHEN "vulnerability_a1" = 1 AND "risk_a1" = 3 THEN 2
  ELSE 0
END
```

7. Click OK and save the layer

### ArcGIS Pro: Python Expression

In ArcGIS Pro's Field Calculator:

1. Create a new Integer field called `combined_a1` / `combined_a2`
2. Switch to the Python tab in Field Calculator
3. Use this code block:

```python
def calc_combined(vulnerability, risk):
    matrix = {
        (3,1): 2, (3,2): 3, (3,3): 3,
        (2,1): 1, (2,2): 2, (2,3): 3,
        (1,1): 1, (1,2): 1, (1,3): 2
    }
    return matrix.get((vulnerability, risk), 0)
```

Then use the expression: `calc_combined(!vulnerability_a1!, !risk_a1!)`

### ArcGIS Pro: ArcPy Script Tool (Alternative)

If you want a more robust solution in ArcGIS Pro:

```python
import arcpy
fc = arcpy.GetParameterAsText(0)  # Input feature class

matrix = {
    (3,1): 2, (3,2): 3, (3,3): 3,
    (2,1): 1, (2,2): 2, (2,3): 3,
    (1,1): 1, (1,2): 1, (1,3): 2
}

arcpy.AddField_management(fc, "combined_a1", "SHORT")

with arcpy.da.UpdateCursor(fc, ["vulnerability_a1", "risk_a1", "combined_a1"]) as cursor:
    for row in cursor:
        v, r = int(row[0]), int(row[1])
        row[2] = matrix.get((v, r), 0)
        cursor.updateRow(row)

arcpy.SetParameterAsText(1, "Complete")
```


> **Note:** You will need to repeat these steps for each asset group. The field names above use the Asset 1 suffix (`_a1`). For Asset 2, replace `_a1` with `_a2` in all field references.

## 14. Final Congratulations

🎉 **You have successfully completed the GIS Vulnerability & Risk Assessment Course!**

You have:
- ✅ Analyzed geospatial data with Python and GeoPandas
- ✅ Applied decision matrices to complex problems
- ✅ Created vulnerability scores from multiple factors
- ✅ Assessed risk through probability and consequence
- ✅ Combined assessments into actionable priorities
- ✅ Visualized results with publication-quality maps
- ✅ Documented methodology for reproducibility

Your combined score map is now ready to:
- Present to decision-makers
- Guide adaptation planning
- Support funding requests
- Inform emergency preparedness
- Drive community resilience

Thank you for completing this course. These skills are valuable for climate adaptation, emergency management, urban planning, and environmental protection.

**Keep learning. Keep mapping. Keep making a difference.**

---

*For questions or to use this data further, contact your instructor or GIS coordinator.*
